# Ninefold Tide — Kaggle Blender CPU/Grease Pencil compatibility probe

Runs once on the existing free Kaggle notebook account, without GPU, asset transfers, downloads, or changes to any existing project. Reports whether the current runtime can build/paint the v09 camera-locked Naya source with Blender 4.3+.


In [ ]:
from pathlib import Path
import json,os,platform,subprocess,sys,shutil,time
from datetime import datetime,timezone

out=Path("/kaggle/working/ninefold_blender_probe")
out.mkdir(parents=True,exist_ok=True)
result={
  "project":"The Ninefold Tide",
  "purpose":"Preflight only, no personal Drive assets transferred and no GPU model downloads",
  "timestamp_utc":datetime.now(timezone.utc).isoformat(),
  "is_private_required":True,
  "runtime":"Kaggle notebook, free CPU instance",
  "python_version":platform.python_version(),
  "os":platform.platform(),
  "gpu_needed":False,
  "user_is_root":os.geteuid()==0,
  "blender_available":False,
  "blender_version":None,
  "blender_grease_pencil_v3_found":False,
  "blender_save_test":False,
  "blender_render_test":False,
  "would_run_v09_source_without_install":False,
  "errors":[],
}
disk=shutil.disk_usage("/kaggle/working")
result["working_disk_free_gib"]=round(disk.free/(1024**3),2)
result["working_disk_total_gib"]=round(disk.total/(1024**3),2)
result["blender_binary"]=shutil.which("blender")
result["ffmpeg_found"]=bool(shutil.which("ffmpeg"))
result["apt_get_found"]=bool(shutil.which("apt-get"))
print("Preflight only; no expensive GPU allocation or art copies.",flush=True)

if result["blender_binary"]:
    command=[result["blender_binary"],"--background","--factory-startup","--python-expr",
      "import bpy,json,os;from pathlib import Path;"
      "p=Path('/kaggle/working/ninefold_blender_probe');"
      "p.mkdir(parents=True,exist_ok=True);"
      "gp=hasattr(bpy.data,'grease_pencils_v3') or hasattr(bpy.types,'GreasePencilv3');"
      "bpy.ops.wm.save_as_mainfile(filepath=str(p/'cpu_smoke.blend'));"
      "scene=bpy.context.scene;"
      "scene.render.engine='BLENDER_EEVEE_NEXT' if hasattr(bpy.app,'version') and bpy.app.version>=(4,2,0) else 'BLENDER_EEVEE';"
      "scene.render.resolution_x=64;scene.render.resolution_y=64;"
      "scene.render.resolution_percentage=100;"
      "scene.render.image_settings.file_format='PNG';"
      "scene.render.filepath=str(p/'smoke_64.png');"
      "bpy.ops.render.render(write_still=True);"
      "(p/'bpy_probe.json').write_text(json.dumps({'version':bpy.app.version_string,'version_tuple':list(bpy.app.version),'gp_v3':gp,'engine':scene.render.engine,'saved_blend':(p/'cpu_smoke.blend').exists(),'rendered_png':(p/'smoke_64.png').exists()},indent=2))"
    ]
    try:
        p=subprocess.run(command,timeout=180,capture_output=True,text=True)
        (out/"blender_console_tail.txt").write_text(
           (p.stdout or "")[-2200:]+"\nSTDERR\n"+(p.stderr or "")[-2200:],encoding="utf8")
        if (out/"bpy_probe.json").exists():
            details=json.loads((out/"bpy_probe.json").read_text())
            result.update({"blender_available":True,"blender_version":details["version"],
                "blender_grease_pencil_v3_found":details["gp_v3"],
                "blender_save_test":details["saved_blend"],
                "blender_render_test":details["rendered_png"]})
            result["would_run_v09_source_without_install"]=bool(
                 details["version_tuple"]>=[4,3,0] and details["gp_v3"]
                 and details["saved_blend"] and details["rendered_png"])
        else:
            result["errors"].append("Blender binary present but failed smoke render; see console tail")
    except Exception as exc:
        result["errors"].append("Blender test failed: "+type(exc).__name__)
else:
    result["errors"].append("Blender binary absent on Kaggle CPU runtime; source v09r1 requires free Blender installation")
    if result["apt_get_found"]:
        try:
            apt=subprocess.run(["apt-cache","policy","blender"],
                timeout=30,capture_output=True,text=True)
            result["apt_blender_candidate"]=(apt.stdout or "")[-600:]
        except Exception:
            pass
(out/"blender_runtime_inventory.json").write_text(json.dumps(result,indent=2)+"\n",encoding="utf8")
print("NINEFOLD_BLENDER_PREFLIGHT",json.dumps({
   "available":result["blender_available"],
   "version":result["blender_version"],
   "grease_pencil_v3":result["blender_grease_pencil_v3_found"],
   "save":result["blender_save_test"],
   "render":result["blender_render_test"],
   "direct_v09":result["would_run_v09_source_without_install"],
   "errors":result["errors"]},indent=2),flush=True)
